In [1]:
!pip install hydra-core lightning torchmetrics wandb omegaconf --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 75.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have num

In [2]:
# Clone the ReGUn repository from GitHub
import subprocess, os

REPO_URL  = "https://github.com/tiensinh2/ReGUn.git"
CODE_DIR  = "/kaggle/working/ReGUn"

if not os.path.isdir(CODE_DIR):
    result = subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, CODE_DIR],
        capture_output=False,
    )
    if result.returncode != 0:
        raise RuntimeError(f"git clone failed with return code {result.returncode}")
    print(f"Cloned {REPO_URL} → {CODE_DIR}")
else:
    print(f"Repository already present at {CODE_DIR}, skipping clone.")

# Confirm key files are present
for f in ["run1_reference.py", "run2_base.py", "run3_unlearning.py", "conf/config.yaml"]:
    path = os.path.join(CODE_DIR, f)
    status = "✓" if os.path.exists(path) else "✗ MISSING"
    print(f"  {status}  {path}")

/kaggle/input/datasets/kiethe/regun-code/ReGUn_visualization.png
/kaggle/input/datasets/kiethe/regun-code/run_slurm.sbatch
/kaggle/input/datasets/kiethe/regun-code/.gitignore
/kaggle/input/datasets/kiethe/regun-code/run2_base.py
/kaggle/input/datasets/kiethe/regun-code/mul_env.def
/kaggle/input/datasets/kiethe/regun-code/README.md
/kaggle/input/datasets/kiethe/regun-code/run1_reference.py
/kaggle/input/datasets/kiethe/regun-code/requirements.txt
/kaggle/input/datasets/kiethe/regun-code/run3_unlearning.py
/kaggle/input/datasets/kiethe/regun-code/utils/utils.py
/kaggle/input/datasets/kiethe/regun-code/utils/__init__.py
/kaggle/input/datasets/kiethe/regun-code/models/resnet.py
/kaggle/input/datasets/kiethe/regun-code/models/swin.py
/kaggle/input/datasets/kiethe/regun-code/models/__init__.py
/kaggle/input/datasets/kiethe/regun-code/models/vit.py
/kaggle/input/datasets/kiethe/regun-code/conf/config.yaml
/kaggle/input/datasets/kiethe/regun-code/conf/model/vit.yaml
/kaggle/input/datasets/kiet

In [3]:
import os

CODE_DIR = "/kaggle/working/ReGUn"

os.environ["CACHE_DIR"]   = "/kaggle/working/cache"
os.environ["DATA_DIR"]    = "/kaggle/working/data"
os.environ["OUTPUTS_DIR"] = "/kaggle/working/outputs"

# Create working directories
os.makedirs("/kaggle/working/cache/models", exist_ok=True)
os.makedirs("/kaggle/working/cache/eval",   exist_ok=True)
os.makedirs("/kaggle/working/outputs",      exist_ok=True)
os.makedirs("/kaggle/working/data",         exist_ok=True)

print(f"CODE_DIR   : {CODE_DIR}")
print(f"CACHE_DIR  : {os.environ['CACHE_DIR']}")
print(f"DATA_DIR   : {os.environ['DATA_DIR']}")
print(f"OUTPUTS_DIR: {os.environ['OUTPUTS_DIR']}")

In [4]:
import os, sys

CODE_DIR = "/kaggle/working/ReGUn"
os.chdir(CODE_DIR)

# Make the repo importable in this process as well
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

print("Working directory:", os.getcwd())

In [5]:
common_overrides = [
    "data=cifar10",
    "model=resnet",
    "model.model.num_classes=10",
    "trainer.accelerator=gpu",    # explicit GPU for Kaggle
    "trainer.devices=1",
    "trainer.precision=16-mixed",
    "logging.offline=true",
]

In [6]:
# ─────────────────────────────────────────────────────────────────
# TEST_MODE: set True to run a fast smoke-test of the whole pipeline.
#   • max_epochs        = 2   (instead of 100 / 10)
#   • limit_train_batches = 5, limit_val_batches = 2
#   • debug_subset_size = 500
#     CIFAR-10 has 50 000 train samples; 500 is ~100× smaller.
#     Must stay ≥ ~112 so that forget split (0.9 × 0.01 × N) ≥ 1 sample.
# Set False for the full production run.
# ─────────────────────────────────────────────────────────────────
TEST_MODE = False      # ← flip to True for a quick sanity-check

# Minimum safe value: ceil(1 / (retain_frac=0.9 × forget_frac=0.01)) = 112
# 500 gives ~4-5 forget samples and keeps the pipeline ~100× faster.
DEBUG_SUBSET = 500

# test_overrides only contains overrides NOT already in common_overrides
# to avoid duplicate keys being passed to Hydra.
test_overrides = [
    "trainer.max_epochs=2",
    "+trainer.limit_train_batches=5",
    "+trainer.limit_val_batches=2",
    f"data.debug_subset_size={DEBUG_SUBSET}",
] if TEST_MODE else []
print("Mode:", "TEST" if TEST_MODE else "FULL")
if TEST_MODE:
    print(f"  debug_subset_size={DEBUG_SUBSET}")
    print(f"  max_epochs=2, limit_train_batches=5, limit_val_batches=2")

Mode: FULL


In [7]:
import subprocess, json, glob, os, pandas as pd
import wandb.proto.wandb_internal_pb2 as pb
from wandb.sdk.internal import datastore

def read_wandb_run(run_dir):
    wandb_files = glob.glob(os.path.join(run_dir, "run-*.wandb"))
    if not wandb_files:
        return None
    store = datastore.DataStore()
    store.open_for_scan(wandb_files[0])
    summary = {}
    while True:
        data = store.scan_data()
        if data is None:
            break
        try:
            record = pb.Record()
            record.ParseFromString(data)
            if record.HasField("summary"):
                for item in record.summary.update:
                    # value_json is a raw JSON string; parse it so we get
                    # Python floats/None instead of the literal string "null"
                    try:
                        summary[item.key] = json.loads(item.value_json)
                    except (json.JSONDecodeError, ValueError):
                        summary[item.key] = item.value_json
        except:
            continue
    return {k: v for k, v in summary.items() if not k.startswith("_")}

def print_summary(summary, title="Results"):
    if not summary:
        print("  (no data)")
        return
    # coerce: non-numeric values (None, dicts, strings) become NaN and are dropped
    s = pd.to_numeric(pd.Series(summary), errors='coerce').dropna()
    # Gom theo prefix (phần trước "/")
    prefixes = sorted(set(k.split("/")[0] for k in s.index if "/" in k))
    print(f"\n{'═'*55}")
    print(f"  {title}")
    print(f"{'═'*55}")
    for prefix in prefixes:
        subset = s[s.index.str.startswith(f"{prefix}/")]
        if subset.empty:
            continue
        print(f"\n  [{prefix}]")
        print(f"  {'─'*50}")
        # Metrics quan trọng lên trước
        priority = ["acc_retain", "acc_forget", "acc_eval", "acc_test",
                    "average_gap", "average_gap_auc", "rmia_auc"]
        printed = set()
        for km in priority:
            key = f"{prefix}/{km}"
            if key in subset.index:
                print(f"  {'★'} {km:<40} {subset[key]:>8.4f}")
                printed.add(key)
        # Còn lại
        rest = subset[~subset.index.isin(printed)]
        for k, v in rest.items():
            metric = k.split("/", 1)[1]
            print(f"    {metric:<42} {v:>8.4f}")
    print(f"{'═'*55}")

CODE_DIR = "/kaggle/working/ReGUn"
for idx in range(1, 5):
    print(f"\n{'='*40}\nReference model {idx}/4\n{'='*40}")
    result = subprocess.run(
        ["python", "run1_reference.py", f"--run-idx={idx}",
         "data=cifar10", "model=resnet", "model.model.num_classes=10",
         "split.forget_frac=0.01",
         "trainer.accelerator=gpu", "trainer.devices=1",
         *test_overrides],
        capture_output=False,
        cwd=CODE_DIR,
    )
    if result.returncode != 0:
        print(f"[ERROR] Reference {idx} failed")
        break
    latest = max(glob.glob("/kaggle/working/outputs/**/wandb/offline-run-*", recursive=True),
                 key=os.path.getmtime)
    summary = read_wandb_run(latest)
    print_summary(summary, title=f"Reference model {idx}/4")


Reference model 1/4


Seed set to 43


[MAIN] Config:
data:
  name: cifar10
  data_dir: ${paths.data_dir}
  download: true
  batch_size: 128
  num_workers: 4
  pin_memory: true
  persistent_workers: true
  prefetch_factor: 2
  debug_subset_size: null
  transforms:
    normalize:
      mean:
      - 0.4914
      - 0.4822
      - 0.4465
      std:
      - 0.247
      - 0.2435
      - 0.2616
    random_crop:
      enabled: true
      size: 32
      padding: 4
    horizontal_flip:
      enabled: true
      p: 0.5
    imagenet_resize:
      enabled: false
      train_size: 224
      eval_resize: 256
      eval_crop: 224
      normalize:
        mean:
        - 0.485
        - 0.456
        - 0.406
        std:
        - 0.229
        - 0.224
        - 0.225
model:
  model:
    name: resnet18
    num_classes: 10
    pretrained: false
    stem: cifar
    freeze_backbone: false
  optim:
    name: sgd
    lr: 0.1
    weight_decay: 0.0005
    momentum: 0.9
    nesterov: true
    betas:
    - 0.9
    - 0.999
  scheduler:
    name: cos

100%|██████████| 170M/170M [00:25<00:00, 6.56MB/s]
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.
wandb: Tracking run with wandb version 0.25.1
wandb: W&B syncing is set to `offline` in this directory. Run `wandb online` or set WANDB_MODE=online to enable cloud syncing.
wandb: Run data is saved locally in /kaggle/working/outputs/2026-07-02/18-00-51/wandb/offline-run-20260702_180123-ibshdobp
Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/model_summary/model_summary.py:242: Precision 16-mixed is not supporte


═══════════════════════════════════════════════════════
  Reference model 1/4
═══════════════════════════════════════════════════════
═══════════════════════════════════════════════════════

Reference model 2/4


...


═══════════════════════════════════════════════════════
  Reference model 2/4
═══════════════════════════════════════════════════════
═══════════════════════════════════════════════════════

Reference model 3/4


Seed set to 45
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.
wandb: Tracking run with wandb version 0.25.1
wandb: W&B syncing is set to `offline` in this directory. Run `wandb online` or set WANDB_MODE=online to enable cloud syncing.
wandb: Run data is saved locally in /kaggle/working/outputs/2026-07-02/18-58-07/wandb/offline-run-20260702_185811-5ltamikj
Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/model_summary/model_summary.py:242: Precision 16-mixed is not supported by the model summary.  Estimated m


═══════════════════════════════════════════════════════
  Reference model 3/4
═══════════════════════════════════════════════════════
═══════════════════════════════════════════════════════

Reference model 4/4


Seed set to 46
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.
wandb: Tracking run with wandb version 0.25.1
wandb: W&B syncing is set to `offline` in this directory. Run `wandb online` or set WANDB_MODE=online to enable cloud syncing.
wandb: Run data is saved locally in /kaggle/working/outputs/2026-07-02/19-26-38/wandb/offline-run-20260702_192642-2i4s6ua3
Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/model_summary/model_summary.py:242: Precision 16-mixed is not supported by the model summary.  Estimated m


═══════════════════════════════════════════════════════
  Reference model 4/4
═══════════════════════════════════════════════════════
═══════════════════════════════════════════════════════


In [8]:
CODE_DIR = "/kaggle/working/ReGUn"
result = subprocess.run(
    ["python", "run2_base.py", "data=cifar10", "model=resnet",
     "split.forget_frac=0.01", "model.model.num_classes=10", *test_overrides],
    capture_output=False,
    cwd=CODE_DIR,
)
latest = max(glob.glob("/kaggle/working/outputs/**/wandb/offline-run-*", recursive=True),
             key=os.path.getmtime)
summary = read_wandb_run(latest)
print_summary(summary, title="Base training")

Seed set to 42
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.
wandb: Tracking run with wandb version 0.25.1
wandb: W&B syncing is set to `offline` in this directory. Run `wandb online` or set WANDB_MODE=online to enable cloud syncing.
wandb: Run data is saved locally in /kaggle/working/outputs/2026-07-02/19-55-07/wandb/offline-run-20260702_195510-in254vte
Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/model_summary/model_summary.py:242: Precision 16-mixed is not supported by the model summary.  Estimated m


═══════════════════════════════════════════════════════
  Base training
═══════════════════════════════════════════════════════

  [base]
  ──────────────────────────────────────────────────
  ★ acc_retain                                 1.0000
  ★ acc_forget                                 1.0000
  ★ acc_eval                                   0.9438
  ★ acc_test                                   0.9343
  ★ average_gap                                0.0454
  ★ average_gap_auc                            0.0398
  ★ rmia_auc                                   0.6290
    forget_entropy                               0.0060
    forget_entropy_gap_retrained                -0.0747
    divergence_retain                            0.0004
    divergence_eval                              0.0229
    divergence_test                              0.0240
    divergence_forget                            0.0309
    smia_loss_acc                                0.4040
    smia_loss_auc                     

In [9]:
# ── Unlearning loop  seed=42  CIFAR-10 ─────────────────
import os, subprocess, glob, collections, json, time
import numpy as np

SEED    = 42
METHODS = ["amun", "finetune", "l1sparse"]

all_results = {}

PRIORITY = [
    "acc_retain", "acc_forget", "acc_eval", "acc_test",
    "average_gap", "average_gap_auc",
    "average_gap_eval", "average_gap_eval_auc",
    "rmia_auc", "rmia_eval_auc",
]

def _fmt(seconds):
    """Format elapsed seconds as HH:MM:SS."""
    h, r = divmod(int(seconds), 3600)
    m, s = divmod(r, 60)
    return f"{h:02d}:{m:02d}:{s:02d}"

cell9_start = time.time()
print(f"\n{'═'*60}")
print(f"  Cell 9 — Unlearning loop started")
print(f"  Methods : {METHODS}")
print(f"  Seed    : {SEED}")
print(f"  Mode    : {'TEST' if TEST_MODE else 'FULL'}")
print(f"{'═'*60}\n")

for method in METHODS:
    t0 = time.time()
    elapsed = lambda: _fmt(time.time() - cell9_start)
    print(f"\n[{elapsed()}] {'='*45}")
    print(f"[{elapsed()}]   Method={method}  Seed={SEED}  [CIFAR-10]")
    print(f"[{elapsed()}] {'='*45}")
    print(f"[{elapsed()}]   Starting subprocess run3_unlearning.py...")

    CODE_DIR = "/kaggle/working/ReGUn"
    result = subprocess.run(
        [
            "python", "run3_unlearning.py",
            f"unlearn={method}",
            f"seed={SEED}",
            "split.forget_frac=0.01",
            *common_overrides,
            *test_overrides,
        ],
        capture_output=False,
        cwd=CODE_DIR,
    )
    method_elapsed = time.time() - t0
    if result.returncode != 0:
        print(f"[{elapsed()}]   [ERROR] method={method} failed after {_fmt(method_elapsed)}, skipping.")
        continue
    print(f"[{elapsed()}]   Subprocess finished in {_fmt(method_elapsed)}. Reading wandb summary...")

    all_runs = glob.glob(
        "/kaggle/working/outputs/**/wandb/offline-run-*",
        recursive=True,
    )
    if not all_runs:
        print(f"[{elapsed()}]   [WARN] No wandb runs found.")
        continue
    latest = max(all_runs, key=os.path.getmtime)
    summary = read_wandb_run(latest)
    if summary is None:
        print(f"[{elapsed()}]   [WARN] No wandb summary found.")
        continue

    all_results[method] = {}
    for key, val in summary.items():
        if key.startswith("unlearning/"):
            try:
                all_results[method][key] = float(val)
            except (ValueError, TypeError):
                pass

    print(f"[{elapsed()}]   Results for [{method.upper()}] (took {_fmt(method_elapsed)}):")
    print(f"  {'─'*55}")
    printed = set()
    for short in PRIORITY:
        key = f"unlearning/{short}"
        if key not in all_results[method]:
            continue
        print(f"  ★ {short:<42} {all_results[method][key]*100:6.2f} %")
        printed.add(key)
    for key in sorted(all_results[method].keys()):
        if key in printed:
            continue
        short = key.split("/", 1)[1]
        print(f"    {short:<44} {all_results[method][key]*100:6.2f} %")

total_elapsed = time.time() - cell9_start
print(f"\n[{_fmt(total_elapsed)}] All methods done. Total time: {_fmt(total_elapsed)}")

# ── Save results ─────────────────────────────────────────
with open("/kaggle/working/results_summary.json", "w") as f:
    json.dump(all_results, f, indent=2)
print(f"[{_fmt(total_elapsed)}] Saved → /kaggle/working/results_summary.json")


  Method=amun  Seed=42  [CIFAR-10]


Seed set to 42
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.
wandb: Tracking run with wandb version 0.25.1
wandb: W&B syncing is set to `offline` in this directory. Run `wandb online` or set WANDB_MODE=online to enable cloud syncing.
wandb: Run data is saved locally in /kaggle/working/outputs/2026-07-02/20-54-12/wandb/offline-run-20260702_205416-kfa74v8n
building AMUN adversarial set: 100%|██████████| 4/4 [00:53<00:00, 13.43s/it]
Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/model_summary/model_summary.p


  [AMUN]
  ───────────────────────────────────────────────────────
  ★ acc_retain                                  99.60 %
  ★ acc_forget                                  86.67 %
  ★ acc_eval                                    92.56 %
  ★ acc_test                                    91.87 %
  ★ average_gap                                 14.34 %
  ★ average_gap_auc                              5.65 %
  ★ average_gap_eval                             7.77 %
  ★ average_gap_eval_auc                         5.73 %
  ★ rmia_auc                                    41.07 %
  ★ rmia_eval_auc                               40.76 %
    average_gap_eval_test                          7.15 %
    average_gap_test                               6.98 %
    divergence_eval                                3.33 %
    divergence_forget                              8.93 %
    divergence_retain                              0.35 %
    divergence_test                                3.50 %
    forget_entropy      

Seed set to 42
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.
wandb: Tracking run with wandb version 0.25.1
wandb: W&B syncing is set to `offline` in this directory. Run `wandb online` or set WANDB_MODE=online to enable cloud syncing.
wandb: Run data is saved locally in /kaggle/working/outputs/2026-07-02/21-07-37/wandb/offline-run-20260702_210741-88pyqbwn
Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/model_summary/model_summary.py:242: Precision 16-mixed is not supported by the model summary.  Estimated m


  [FINETUNE]
  ───────────────────────────────────────────────────────
  ★ acc_retain                                 100.00 %
  ★ acc_forget                                 100.00 %
  ★ acc_eval                                    94.27 %
  ★ acc_test                                    93.42 %
  ★ average_gap                                  6.21 %
  ★ average_gap_auc                              3.87 %
  ★ average_gap_eval                            11.41 %
  ★ average_gap_eval_auc                         3.93 %
  ★ rmia_auc                                    62.44 %
  ★ rmia_eval_auc                               62.53 %
    average_gap_eval_test                          5.30 %
    average_gap_test                               5.18 %
    divergence_eval                                2.34 %
    divergence_forget                              3.08 %
    divergence_retain                              0.04 %
    divergence_test                                2.46 %
    forget_entropy  

Seed set to 42
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.
wandb: Tracking run with wandb version 0.25.1
wandb: W&B syncing is set to `offline` in this directory. Run `wandb online` or set WANDB_MODE=online to enable cloud syncing.
wandb: Run data is saved locally in /kaggle/working/outputs/2026-07-02/21-17-33/wandb/offline-run-20260702_211737-jpfy51j0
Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/model_summary/model_summary.py:242: Precision 16-mixed is not supported by the model summary.  Estimated m


  [L1SPARSE]
  ───────────────────────────────────────────────────────
  ★ acc_retain                                 100.00 %
  ★ acc_forget                                 100.00 %
  ★ acc_eval                                    94.09 %
  ★ acc_test                                    93.02 %
  ★ average_gap                                  5.33 %
  ★ average_gap_auc                              4.50 %
  ★ average_gap_eval                             7.60 %
  ★ average_gap_eval_auc                         4.61 %
  ★ rmia_auc                                    64.55 %
  ★ rmia_eval_auc                               65.06 %
    average_gap_eval_test                          6.66 %
    average_gap_test                               6.43 %
    divergence_eval                                2.35 %
    divergence_forget                              2.94 %
    divergence_retain                              0.11 %
    divergence_test                                2.52 %
    forget_entropy  

In [ ]:
# ══════════════════════════════════════════════════════════════════
# Vector-space embedding  +  per-method sample analysis
# ══════════════════════════════════════════════════════════════════
# For each unlearning method this cell:
#  1. Embeds ALL test samples into penultimate-layer feature space
#     using the base model (computed once, reused for all methods).
#  2. Runs each unlearning method IN-PROCESS (no checkpoint needed)
#     to obtain the unlearned model, then gets its predictions.
#  3. Classifies every sample into 4 flag types:
#     Type-A  wrong classified by unlearn model
#     Type-B  classified differently between base vs unlearn
#     Type-C  classified differently between retrain vs unlearn
#     Type-D  classified differently between base vs retrain
#  4. For each flagged set, finds its k-NN neighbours in vector space.
#  5. For every group (flagged + neighbours), logs the class-prediction
#     distribution from base / retrain / unlearn models.
#  6. Saves results to JSON under /kaggle/working/.
#
# NOTE: The config for this cell NEVER uses debug_subset_size so the
# full 10 000-sample test set is always used for the embedding space.
# base/retrain predictions are computed once and reused across all methods.
# ══════════════════════════════════════════════════════════════════

import os, sys, json, time
from copy import deepcopy
import numpy as np
import torch
import torch.nn.functional as F
from collections import Counter
from torch.utils.data import DataLoader

# ── Timing helper ──────────────────────────────────────────────────
def _fmt(seconds):
    """Format elapsed seconds as HH:MM:SS."""
    h, r = divmod(int(seconds), 3600)
    m, s = divmod(r, 60)
    return f"{h:02d}:{m:02d}:{s:02d}"

_cell_start = time.time()
def _ts():
    """Return elapsed wall-clock string since cell start."""
    return _fmt(time.time() - _cell_start)

print(f"\n{'═'*65}")
print(f"  Cell: Vector-space analysis  started")
print(f"  Methods : {METHODS}")
print(f"{'═'*65}\n")

CODE_DIR = "/kaggle/working/ReGUn"
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)
os.chdir(CODE_DIR)

from hydra import compose, initialize_config_dir
from hydra.core.global_hydra import GlobalHydra
from omegaconf import OmegaConf
from models import load_model
from data import build_datamodule
from mul import run_mul_strategy
from utils import seed_everything
import lightning.pytorch as pl

# ── Compact per-epoch progress callback ────────────────────────────
# Prints one line per epoch so you can track progress without
# the noisy Lightning banners or full progress bars.
class _EpochPrintCallback(pl.Callback):
    def __init__(self, method_name, cell_start):
        self._method = method_name
        self._cell_start = cell_start
        self._epoch_start = None
    def on_train_epoch_start(self, trainer, pl_module):
        self._epoch_start = time.time()
        ep   = trainer.current_epoch + 1
        maxe = trainer.max_epochs
        ts   = _fmt(time.time() - self._cell_start)
        print(f"[{ts}]   [{self._method}] epoch {ep}/{maxe} started", flush=True)
    def on_train_epoch_end(self, trainer, pl_module):
        ep   = trainer.current_epoch + 1
        maxe = trainer.max_epochs
        ts   = _fmt(time.time() - self._cell_start)
        dur  = _fmt(time.time() - self._epoch_start) if self._epoch_start else "?"
        # Pull logged metrics if available
        cb_metrics = trainer.callback_metrics
        parts = []
        for k in ["retain/acc", "train/acc", "val/acc", "retain/loss", "train/loss", "val/loss"]:
            if k in cb_metrics:
                parts.append(f"{k}={float(cb_metrics[k]):.4f}")
        metrics_str = "  ".join(parts) if parts else ""
        print(f"[{ts}]   [{self._method}] epoch {ep}/{maxe} done (took {dur})  {metrics_str}", flush=True)

# ── Config builder — NO debug_subset_size ever ─────────────────────
def _build_cfg(extra_overrides=None):
    """Build a Hydra config for cifar10/resnet/seed=42.
    Never passes debug_subset_size so the full test set is always used.
    """
    GlobalHydra.instance().clear()
    base = [
        "data=cifar10",
        "model=resnet",
        "model.model.num_classes=10",
        "trainer.accelerator=gpu",
        "trainer.devices=1",
        "split.forget_frac=0.01",
        "logging.offline=true",
    ]
    with initialize_config_dir(
        config_dir=os.path.join(CODE_DIR, "conf"),
        version_base=None,
    ):
        cfg = compose(config_name="config", overrides=base + (extra_overrides or []))
    OmegaConf.update(cfg, "paths.cache_dir",   os.environ.get("CACHE_DIR",   "cache"), merge=True)
    OmegaConf.update(cfg, "paths.data_dir",    os.environ.get("DATA_DIR",    "cache"), merge=True)
    OmegaConf.update(cfg, "paths.outputs_dir", os.environ.get("OUTPUTS_DIR", "outputs"), merge=True)
    return cfg


DEVICE          = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED            = 42
K_NEIGHBORS     = 10      # max neighbours for aggregate pool
K_PER_SAMPLE    = 5       # max neighbours per flagged sample (with threshold)
DIST_THRESHOLD  = 0.15    # cosine-distance threshold: keep neighbour only if dist < this
#   cosine distance = 1 - cosine_similarity, so dist < 0.15 means sim > 0.85
BATCH_SIZE      = 256
NUM_CLASSES     = 10
print(f"[{_ts()}] Device : {DEVICE}")

# ── Base config + datamodule (full dataset, no subset) ─────────────
print(f"[{_ts()}] Building config and datamodule...")
cfg_base = _build_cfg()
dm       = build_datamodule(cfg_base)

BASE_CKPT    = str(cfg_base.run.base_weights)
RETRAIN_CKPT = str(cfg_base.run.retrain_weights)
print(f"[{_ts()}] Base    ckpt : {BASE_CKPT}")
print(f"[{_ts()}] Retrain ckpt : {RETRAIN_CKPT}")

print(f"[{_ts()}] Loading base model...")
model_base    = load_model(cfg_base, BASE_CKPT).to(DEVICE).eval()
print(f"[{_ts()}] Loading retrain model...")
model_retrain = load_model(cfg_base, RETRAIN_CKPT).to(DEVICE).eval()

# ── Full test-set DataLoader (never shrunk) ────────────────────────
eval_loader = DataLoader(
    dm.ds_test,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=(DEVICE.type == "cuda"),
)
N_TOTAL = len(dm.ds_test)
print(f"[{_ts()}] Test set size: {N_TOTAL}")


# ── Feature extractor (penultimate avgpool layer of ResNet18) ───────
def extract_features_and_preds(model, loader, device):
    """Return (features [N,512], preds [N], labels [N]).
    Features are L2-normalised avgpool embeddings.
    """
    inner    = model.model
    pool_out = []
    handle   = inner.avgpool.register_forward_hook(
        lambda m, i, o: pool_out.append(o.detach().flatten(1))
    )
    feats, preds, labels = [], [], []
    model.eval()
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            pool_out.clear()
            logits = model(x)
            feats.append(F.normalize(pool_out[0].cpu(), p=2, dim=1))
            preds.append(logits.argmax(dim=1).cpu())
            labels.extend(y.tolist())
    handle.remove()
    return (
        torch.cat(feats).numpy(),
        torch.cat(preds).numpy(),
        np.array(labels),
    )


def extract_softmax(model, loader, device):
    """Return softmax probabilities [N, num_classes] as float32 numpy array."""
    probs_list = []
    model.eval()
    with torch.no_grad():
        for x, _ in loader:
            x = x.to(device)
            logits = model(x)
            probs_list.append(F.softmax(logits.float(), dim=1).cpu())
    return torch.cat(probs_list).numpy()   # [N, C]


def knn_per_sample(sim_matrix, query_idx, k, dist_threshold=0.0):
    """Return up to k neighbour indices for a single query sample.
    Skips any neighbour whose cosine distance (1 - sim) > dist_threshold.
    Results are sorted by distance ascending (closest first).
    Returns list of (idx, distance) tuples.
    """
    row = sim_matrix[query_idx].copy()
    row[query_idx] = -np.inf          # exclude self
    k_eff = min(k, len(row) - 1)
    top   = np.argpartition(row, -k_eff)[-k_eff:]      # top-k by sim
    top   = top[np.argsort(row[top])[::-1]]             # sort by sim desc
    # Convert to distance and filter: keep only if distance < dist_threshold
    result = [(int(idx), round(1.0 - float(row[idx]), 5))
              for idx in top
              if (1.0 - float(row[idx])) < dist_threshold]
    return result   # [(idx, dist), ...] closest first


# ── k-NN helpers ────────────────────────────────────────────────────
def build_knn_index(features_np, device=None):
    """Return (N,N) cosine-similarity matrix; matmul on GPU when possible."""
    F_t = torch.from_numpy(features_np)
    if device is not None and device.type != "cpu":
        F_t = F_t.to(device)
    return torch.mm(F_t, F_t.t()).cpu().numpy()


def knn_indices(sim_matrix, query_indices, k, exclude_self=True):
    """Return unique neighbour indices for all query_indices.
    For each row i, the 'self' column to exclude is query_indices[i].
    If k >= N_TOTAL the argpartition is clamped to avoid index error.
    """
    if len(query_indices) == 0:
        return np.array([], dtype=int)
    rows = sim_matrix[query_indices].copy()   # shape (Q, N)
    if exclude_self:
        # For row i, exclude column query_indices[i] (the sample itself).
        rows[np.arange(len(query_indices)), query_indices] = -np.inf
    k_eff = min(k, rows.shape[1] - 1)
    top_k = np.argpartition(rows, -k_eff, axis=1)[:, -k_eff:]
    # Remove any self-indices that may survive due to ties
    mask = top_k != np.array(query_indices)[:, None]
    neighbors = top_k[mask]
    return np.unique(neighbors)


# ── Distribution helpers ────────────────────────────────────────────
def mean_softmax(softmax_arr, idx):
    """Mean softmax over a set of samples.
    Returns plain list [p0, p1, ..., p9] in class order 0→9.
    softmax_arr: [N, C] numpy array.
    idx: array/list of sample indices.
    """
    if len(idx) == 0:
        return []
    mean = softmax_arr[idx].mean(axis=0)   # [C]
    return [round(float(v), 5) for v in mean]

def gt_dist(labels, idx, nc=NUM_CLASSES):
    """Return ground-truth label counts in class order 0→9."""
    c = Counter(int(v) for v in labels[idx])
    return [c.get(i, 0) for i in range(nc)]


# ══════════════════════════════════════════════════════════════════
# Step 1 – Base and retrain embeddings/predictions (computed once)
# ══════════════════════════════════════════════════════════════════
print(f"\n[{_ts()}] ── Step 1: Embed base model + build k-NN index ──────────")
_t = time.time()
print(f"[{_ts()}]   Extracting base-model embeddings + predictions...")
feats_base, preds_base, labels_all = extract_features_and_preds(model_base, eval_loader, DEVICE)
print(f"[{_ts()}]   Feature shape : {feats_base.shape}")
print(f"[{_ts()}]   Base accuracy : {(preds_base == labels_all).mean():.4f}  (took {_fmt(time.time()-_t)})")

_t = time.time()
print(f"[{_ts()}]   Extracting retrain-model predictions + softmax...")
_, preds_retrain, _ = extract_features_and_preds(model_retrain, eval_loader, DEVICE)
print(f"[{_ts()}]   Retrain accuracy : {(preds_retrain == labels_all).mean():.4f}  (took {_fmt(time.time()-_t)})")

_t = time.time()
print(f"[{_ts()}]   Extracting base softmax probabilities...")
softmax_base    = extract_softmax(model_base,    eval_loader, DEVICE)   # [N, C]
print(f"[{_ts()}]   Extracting retrain softmax probabilities...")
softmax_retrain = extract_softmax(model_retrain, eval_loader, DEVICE)   # [N, C]
print(f"[{_ts()}]   Softmax extraction done  (took {_fmt(time.time()-_t)})")

_t = time.time()
print(f"[{_ts()}]   Building k-NN similarity matrix ({feats_base.shape[0]}x{feats_base.shape[0]})...")
sim_matrix = build_knn_index(feats_base, device=DEVICE)
print(f"[{_ts()}]   Similarity matrix shape : {sim_matrix.shape}  (took {_fmt(time.time()-_t)})")


# ══════════════════════════════════════════════════════════════════
# Step 2 – Per-method analysis
# For each method: run unlearning in-process, then analyse predictions.
# base/retrain preds already computed above — NOT recomputed per method.
# ══════════════════════════════════════════════════════════════════
from mul import MULEvaluator

# Validate the eval cache against the current forget-set size.
# If it was built with debug_subset_size the shapes will be wrong;
# delete it so MULEvaluator rebuilds it from the full dataset.
_cache_path = str(cfg_base.mul_eval.cache)
if os.path.exists(_cache_path):
    _cached = torch.load(_cache_path, map_location="cpu", weights_only=False)
    _ref_probs = _cached.get("reference_probs", {})
    _forget_probs = _ref_probs.get("forget", [])
    _expected_n_forget = len(dm.eval_forget)   # current full-dataset forget size
    _cached_n_forget = len(_forget_probs[0]) if _forget_probs else 0
    if _cached_n_forget != _expected_n_forget:
        os.remove(_cache_path)
        print(f"[{_ts()}] Removed mismatched cache "
              f"(cached={_cached_n_forget}, expected={_expected_n_forget}): {_cache_path}")
    else:
        print(f"[{_ts()}] Cache valid (forget size={_cached_n_forget}), reusing.")

# Close any wandb run left open by the unlearning subprocess cells
# so it doesn't get reused or duplicated during in-process runs.
import wandb as _wandb
if _wandb.run is not None:
    _wandb.finish()

# Build the evaluator once — shared by all methods.
# evaluate_each_epoch=true in every method yaml means the strategy always
# attaches UnlearningEpochEvaluationCallback, which requires a real evaluator.
print(f"[{_ts()}] Building shared MULEvaluator (may load reference cache)...")
_t = time.time()
shared_evaluator = MULEvaluator(cfg=cfg_base, datamodule=dm)
print(f"[{_ts()}] MULEvaluator ready  (took {_fmt(time.time()-_t)})")

vector_analysis_results = {}

for method in METHODS:
    _method_start = time.time()
    print(f"\n[{_ts()}] {'═'*58}")
    print(f"[{_ts()}]   Step 2: Vector Analysis  method={method}")
    print(f"[{_ts()}] {'═'*58}")

    # Config for in-process unlearning:
    #   trainer.enable_progress_bar=false -> suppress Lightning progress bar
    #   unlearn.evaluate_each_epoch=false -> skip per-epoch MIA eval
    #     (saves ~10 min/method; the vector analysis does its own eval)
    # NOTE: logging.enable is left at its default (true) so that
    #   build_trainer receives a real logger — a silent CSVLogger is passed
    #   as wandb_run so LearningRateMonitor doesn't crash.
    print(f"[{_ts()}]   Building config for {method} (evaluate_each_epoch=false)...")
    cfg_method = _build_cfg(extra_overrides=[
        f"unlearn={method}",
        "trainer.enable_progress_bar=false",
        "+trainer.enable_model_summary=false",
        "unlearn.evaluate_each_epoch=false",
    ])
    seed_everything(SEED)

    # Use a silent CSVLogger as dummy logger so LearningRateMonitor
    # (always added by build_trainer) has a logger to talk to,
    # without spawning a wandb run.
    from lightning.pytorch.loggers import CSVLogger as _CSVLogger
    _dummy_logger = _CSVLogger(
        save_dir="/kaggle/working/csv_logs",
        name=f"vec_{method}",
        flush_logs_every_n_steps=10000,
    )

    # Re-load a fresh copy of the base model to pass to the strategy
    print(f"[{_ts()}]   Loading fresh base model...")
    model_for_unlearn = load_model(cfg_base, BASE_CKPT)

    # Inject the epoch-print callback via a monkey-patch on new_trainer.
    # run_mul_strategy calls strategy.new_trainer() internally, so we
    # wrap it to append our callback without modifying source code.
    from copy import deepcopy as _deepcopy
    from mul import _MUL_STRATEGIES
    _epoch_cb = _EpochPrintCallback(method, _cell_start)
    _strategy_cls = _MUL_STRATEGIES[method]
    _orig_new_trainer = _strategy_cls.new_trainer
    def _patched_new_trainer(self, additional_callbacks=None):
        cbs = list(additional_callbacks or [])
        cbs.append(_epoch_cb)
        return _orig_new_trainer(self, additional_callbacks=cbs)
    _strategy_cls.new_trainer = _patched_new_trainer

    # Run the unlearning strategy in-process (no subprocess, no checkpoint)
    print(f"[{_ts()}]   Running {method} unlearning in-process...")
    _t = time.time()
    try:
        model_unlearn = run_mul_strategy(
            cfg=cfg_method,
            base_model=model_for_unlearn,
            dm=dm,
            wandb_run=_dummy_logger,
            evaluator=shared_evaluator,
        ).to(DEVICE).eval()
    finally:
        # Always restore the original new_trainer
        _strategy_cls.new_trainer = _orig_new_trainer
    print(f"[{_ts()}]   Unlearning done  (took {_fmt(time.time()-_t)})")

    # Predictions + softmax on the full test set
    _t = time.time()
    print(f"[{_ts()}]   Extracting unlearned-model predictions + softmax...")
    _, preds_unlearn, _ = extract_features_and_preds(model_unlearn, eval_loader, DEVICE)
    softmax_unlearn = extract_softmax(model_unlearn, eval_loader, DEVICE)   # [N, C]
    unlearn_acc = (preds_unlearn == labels_all).mean()
    print(f"[{_ts()}]   Unlearn accuracy : {unlearn_acc:.4f}  (took {_fmt(time.time()-_t)})")

    # ── Flag sets ─────────────────────────────────────────────────
    # Type-A : wrong classified by unlearn model
    idx_type_A = np.where(preds_unlearn != labels_all)[0]

    # Type-B : classified differently between base and unlearn
    idx_type_B = np.where(preds_base != preds_unlearn)[0]

    # Type-C : classified differently between retrain and unlearn
    idx_type_C = np.where(preds_retrain != preds_unlearn)[0]

    # Type-D : classified differently between base and retrain
    #   (constant across methods — same base/retrain predictions always)
    idx_type_D = np.where(preds_base != preds_retrain)[0]

    flag_sets = {
        "type_A_wrong_unlearn":      idx_type_A,
        "type_B_base_vs_unlearn":    idx_type_B,
        "type_C_retrain_vs_unlearn": idx_type_C,
        "type_D_base_vs_retrain":    idx_type_D,
    }

    print(f"[{_ts()}]   Flag set sizes:")
    for t, idx_t in flag_sets.items():
        print(f"[{_ts()}]     {t:<44} : {len(idx_t):5d} samples  "
              f"({100*len(idx_t)/N_TOTAL:.1f} %)")

    # ── Per-type analysis ─────────────────────────────────────────
    method_result = {"unlearn_acc": float(unlearn_acc), "groups": {}}

    for type_name, flagged_idx in flag_sets.items():
        if len(flagged_idx) == 0:
            print(f"[{_ts()}]     {type_name} — 0 flagged samples, skipping.")
            method_result["groups"][type_name] = {
                "n_flagged": 0, "n_neighbors": 0, "n_group": 0,
                "flagged_indices": [], "neighbor_indices": [],
                "class_distributions": {}, "samples": [],
            }
            continue

        # ── Aggregate k-NN (all flagged → unique neighbour pool) ──
        _t = time.time()
        neighbor_idx = knn_indices(
            sim_matrix, flagged_idx, k=K_NEIGHBORS, exclude_self=True
        )
        group_idx = np.unique(np.concatenate([flagged_idx, neighbor_idx]))
        print(f"[{_ts()}]     {type_name}: flagged={len(flagged_idx)}  "
              f"pool_neighbours={len(neighbor_idx)}  group={len(group_idx)}  "
              f"(kNN took {_fmt(time.time()-_t)})")

        # ── Mean-softmax distribution over each group ─────────────
        # Each entry: mean softmax probability per class over all samples
        # in the group, sorted by probability descending.
        def _agg_dist(idx):
            arr = np.array(idx, dtype=int) if not isinstance(idx, np.ndarray) else idx
            return {
                "base":         mean_softmax(softmax_base,    arr),
                "retrain":      mean_softmax(softmax_retrain, arr),
                "unlearn":      mean_softmax(softmax_unlearn, arr),
                "ground_truth": gt_dist(labels_all,           arr),
            }
        dist_flagged   = _agg_dist(flagged_idx)
        dist_neighbors = _agg_dist(neighbor_idx) if len(neighbor_idx) > 0 else _agg_dist(np.array([], dtype=int))
        dist_group     = _agg_dist(group_idx)

        # ── Per-sample records: each flagged sample + its top-5 neighbours ─
        # Each record stores sample index, gt label, argmax predictions,
        # and full softmax vector from all 3 models.
        _t = time.time()
        samples_data = []
        n_empty = 0   # count samples with 0 neighbours above threshold
        for fi in flagged_idx:
            nb_pairs = knn_per_sample(
                sim_matrix, int(fi),
                k=K_PER_SAMPLE, dist_threshold=DIST_THRESHOLD,
            )  # list of (idx, dist), filtered to dist <= DIST_THRESHOLD
            nb_records = []
            for ni, dist in nb_pairs:
                nb_records.append({
                    "idx":                  ni,
                    "dist":                 dist,
                    "gt":                   int(labels_all[ni]),
                    "pred_base":            int(preds_base[ni]),
                    "pred_retrain":         int(preds_retrain[ni]),
                    "pred_unlearn":         int(preds_unlearn[ni]),
                    "class_dist_base":      mean_softmax(softmax_base,    [ni]),
                    "class_dist_retrain":   mean_softmax(softmax_retrain, [ni]),
                    "class_dist_unlearn":   mean_softmax(softmax_unlearn, [ni]),
                })
            if not nb_records:
                n_empty += 1
            samples_data.append({
                "idx":                  int(fi),
                "gt":                   int(labels_all[fi]),
                "pred_base":            int(preds_base[fi]),
                "pred_retrain":         int(preds_retrain[fi]),
                "pred_unlearn":         int(preds_unlearn[fi]),
                "class_dist_base":      mean_softmax(softmax_base,    [fi]),
                "class_dist_retrain":   mean_softmax(softmax_retrain, [fi]),
                "class_dist_unlearn":   mean_softmax(softmax_unlearn, [fi]),
                "neighbors":            nb_records,
            })
        print(f"[{_ts()}]     Per-sample records: {len(samples_data)}  "
              f"(dist_threshold={DIST_THRESHOLD}, {n_empty} had 0 qualifying neighbours)  "
              f"(took {_fmt(time.time()-_t)})")

        method_result["groups"][type_name] = {
            "n_flagged":           int(len(flagged_idx)),
            "n_neighbors":         int(len(neighbor_idx)),
            "n_group":             int(len(group_idx)),
            "flagged_indices":     flagged_idx.tolist(),
            "neighbor_pool_indices": neighbor_idx.tolist(),
            # Aggregate class-distribution counts over flagged / neighbour-pool / group
            "class_distributions": {
                "flagged":        dist_flagged,
                "neighbor_pool":  dist_neighbors,
                "group":          dist_group,
            },
            # Per-sample: each flagged sample with its own top-5 neighbours
            # and full softmax from all 3 models
            "samples": samples_data,
        }

        # ── Pretty-print aggregate distribution ──────────────────
        print(f"\n[{_ts()}]   ── {type_name} ── aggregate class distribution")
        for scope, dist_d in [("flagged",      dist_flagged),
                               ("neighbor_pool", dist_neighbors),
                               ("group",         dist_group)]:
            print(f"[{_ts()}]     [{scope}]")
            gt = dist_d["ground_truth"]
            for model_name, dist in [("base",    dist_d["base"]),
                                      ("retrain", dist_d["retrain"]),
                                      ("unlearn", dist_d["unlearn"])]:
                top_str = "  ".join(f"c{c}:{p:.3f}" for c, p in enumerate(dist))
                print(f"[{_ts()}]       {model_name:<8}  {top_str}")
            top_gt_str = "  ".join(f"c{c}:{n}" for c, n in enumerate(gt))
            print(f"[{_ts()}]       {'gt':<8}  {top_gt_str}")

        # ── Log sample count summary (full data saved to JSON) ───
        n_with_nb = sum(1 for r in samples_data if r["neighbors"])
        avg_nb    = sum(len(r["neighbors"]) for r in samples_data) / max(len(samples_data), 1)
        print(f"[{_ts()}]   {type_name}: {len(samples_data)} samples saved, "
              f"{n_with_nb} have >=1 neighbour (dist<{DIST_THRESHOLD}), "
              f"avg {avg_nb:.1f} neighbours each")

    vector_analysis_results[method] = method_result
    print(f"\n[{_ts()}]   Method '{method}' complete  (method total: {_fmt(time.time()-_method_start)})")


# ══════════════════════════════════════════════════════════════════
# Step 3 – Persist results
# ══════════════════════════════════════════════════════════════════
_t = time.time()
print(f"\n[{_ts()}] ── Step 3: Saving results ─────────────────────────────────")
OUT_PATH = "/kaggle/working/vector_analysis_results.json"
with open(OUT_PATH, "w") as f:
    json.dump(vector_analysis_results, f, indent=2)
print(f"[{_ts()}] Saved vector analysis → {OUT_PATH}  (took {_fmt(time.time()-_t)})")

# ── Compact per-method summary table ──────────────────────────────
print("\n" + "═"*80)
print("  SUMMARY: flagged sample counts per method and type")
print("  Type-A = wrong by unlearn  | Type-B = base vs unlearn")
print("  Type-C = retrain vs unlearn | Type-D = base vs retrain (constant)")
print("═"*80)
header_types = ["type_A", "type_B", "type_C", "type_D"]
key_map = {
    "type_A": "type_A_wrong_unlearn",
    "type_B": "type_B_base_vs_unlearn",
    "type_C": "type_C_retrain_vs_unlearn",
    "type_D": "type_D_base_vs_retrain",
}
print(f"  {'Method':<12}" + "".join(f"  {t:<12}" for t in header_types))
print("  " + "─"*64)
for method, mres in vector_analysis_results.items():
    row = f"  {method:<12}"
    for t in header_types:
        n = mres["groups"].get(key_map[t], {}).get("n_flagged", 0)
        row += f"  {n:<12}"
    print(row)
print("═"*80)
print(f"\n[{_ts()}] Cell complete — total wall time: {_fmt(time.time()-_cell_start)}")